# 05 · Grupos que aparecen solos en el mapa (HDBSCAN)

Hasta ahora evaluamos los mapas contra las categorías de YouTube. Pero esas categorías son amplias: *Arts & Entertainment* junta conciertos, baile, animación y cine. Aquí damos vuelta la pregunta: **¿qué grupos aparecen en el mapa sin decirle nada sobre categorías?**

Pasos:

1. Agrupamos el mapa 3D de la web con **HDBSCAN**, que busca zonas densas y deja los puntos sueltos como "ruido".
2. Calculamos el **centroide** de cada grupo en el espacio original de 768 dimensiones.
3. **Bautizamos** cada grupo con las palabras más cercanas a su centroide (zero-shot, como en el notebook 04).
4. Comparamos los grupos con las categorías de YouTube.

Usamos el mapa vigente de la web (`data/maps/`, representación de texto). La web usa este mismo procedimiento (`recolect/clusters.py`, con los mismos parámetros) para su botón **Colorear por → HDBSCAN**, así que los grupos de aquí son los que ve alguien al explorar.

In [ ]:
import sys
from pathlib import Path

# Los notebooks viven en notebooks/, pero los datos (data/) y el código (analysis/, recolect/)
# están en la raíz del repo. Así funcionan abriéndolos desde cualquiera de las dos carpetas.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT))

In [ ]:
import json
from collections import defaultdict

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from sklearn.cluster import HDBSCAN
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

from analysis.metricas import DATA, cargar_videos, cargar_vocabulario, matriz
from analysis.graficos import INK, MUTED, estilo, plot3d

estilo()

In [ ]:
version = json.loads((DATA / "maps" / "current.json").read_text())["version"]
coords = pd.read_parquet(DATA / "maps" / version / "coords.parquet")

# videos de YT-8M con su posición en el mapa; los recibidos por la web no tienen categoría, así que quedan fuera
videos = cargar_videos(solo_completos=False).drop_duplicates("youtube_id")
df = coords[["youtube_id", "x", "y", "z"]].merge(videos, on="youtube_id")

X3 = df[["x", "y", "z"]].to_numpy(dtype=np.float64)    # posición en el mapa 3D
X = matriz(df, "text")                                   # vector original de 768-d
y = df["vertical"].to_numpy(dtype=str)
print(f"mapa {version}: {len(df):,} videos")

## 1. HDBSCAN sobre el mapa

Dos parámetros importan:

- `min_cluster_size`: el tamaño mínimo de un grupo. Lo subimos a 220 para quedarnos con temas grandes y legibles.
- `min_samples`: qué tan exigente es con la densidad. Más alto = más puntos quedan como ruido (-1).

In [ ]:
grupos = HDBSCAN(min_cluster_size=220, min_samples=30).fit_predict(X3)
K = grupos.max() + 1
print(f"{K} grupos · {np.mean(grupos == -1):.0%} de los videos quedan como ruido")

## 2. Centroides y asignación del ruido

El centroide de un grupo es el promedio de sus vectores, normalizado: la "dirección media" del tema en el espacio de 768-d.

$$\mu_k = \frac{\sum_{i \in C_k} x_i}{\left\lVert \sum_{i \in C_k} x_i \right\rVert}$$

Para que todos los videos tengan grupo, cada punto de ruido se asigna al centroide más parecido.

In [ ]:
centroides = np.stack([X[grupos == k].mean(axis=0) for k in range(K)])
centroides /= np.linalg.norm(centroides, axis=1, keepdims=True)

asignado = grupos.copy()
ruido = grupos == -1
asignado[ruido] = np.argmax(X[ruido] @ centroides.T, axis=1)

## 3. ¿Se parecen a las categorías de YouTube?

- **Pureza** de un grupo: qué fracción de sus videos es de su categoría más común. La global es el promedio ponderado por tamaño.
- **NMI** y **ARI**: qué tan parecidas son las dos particiones en conjunto (0 = nada, 1 = idénticas).

Una pureza alta con NMI moderado es buena señal: quiere decir que los grupos **subdividen** las categorías (varios grupos dentro de *Games*, por ejemplo) en vez de mezclarlas.

In [ ]:
valido = y != "(Unknown)"
tabla = pd.crosstab(asignado[valido], y[valido])        # grupos × categorías
pureza = tabla.max(axis=1) / tabla.sum(axis=1)

print(f"pureza global: {tabla.max(axis=1).sum() / tabla.values.sum():.3f}")
print(f"NMI:           {normalized_mutual_info_score(y[valido], asignado[valido]):.3f}")
print(f"ARI:           {adjusted_rand_score(y[valido], asignado[valido]):.3f}")

## 4. Bautizo zero-shot

Para nombrar cada grupo comparamos su centroide con dos vocabularios:

1. **Las 3.806 etiquetas de YT-8M** (`emb_vocab.parquet`), embebidas como texto.
2. **Una folksonomía**: los tags que los propios creadores escriben al subir el video. Para no tener que pasar cada tag por el modelo, el vector de un tag es el centroide de los videos que lo usan (misma fórmula de arriba). Nos quedamos con los tags que aparecen en 8–800 videos: menos es ruido, más son palabras genéricas.

In [ ]:
GENERICOS = {"video", "videos", "youtube", "new", "the", "and", "for", "with", "how", "live", "full", "free",
             "hd", "official", "part", "episode", "best", "top", "clip", "show"}

filas_de = defaultdict(list)          # tag -> filas de los videos que lo usan
for i, tags in enumerate(df["tags"]):
    if not isinstance(tags, (list, np.ndarray)):
        continue
    for t in {str(t).strip().lower() for t in tags}:
        if 3 <= len(t) <= 28 and t not in GENERICOS and not t.isdigit():
            filas_de[t].append(i)
filas_de = {t: f for t, f in filas_de.items() if 8 <= len(f) <= 800}

folk_nombres = np.array(list(filas_de))
folk = np.stack([X[f].mean(axis=0) for f in filas_de.values()])
folk /= np.linalg.norm(folk, axis=1, keepdims=True)
print(f"{len(folk_nombres):,} tags frecuentes")

In [ ]:
vocab = cargar_vocabulario()
L = np.stack(vocab["emb"]).astype(np.float32)
nombres = vocab["Name"].to_numpy()

def bautizo(k):
    """2 etiquetas de YT-8M + 2 tags de la folksonomía más cercanos al centroide, sin repetir."""
    candidatas = [*nombres[np.argsort(-(L @ centroides[k]))[:2]],
                  *folk_nombres[np.argsort(-(folk @ centroides[k]))[:2]]]
    palabras = []
    for w in candidatas:
        if w.lower() not in {p.lower() for p in palabras}:
            palabras.append(w)
    return " · ".join(palabras[:3])

resumen = pd.DataFrame({
    "nombre": [bautizo(k) for k in range(K)],
    "videos": np.bincount(asignado, minlength=K),
    "categoría dominante": tabla.idxmax(axis=1).reindex(range(K)).to_numpy(),
    "pureza": pureza.reindex(range(K)).round(3).to_numpy(),
}).sort_values("videos", ascending=False)
resumen.to_csv(DATA / "hdbscan_centroids.csv", index_label="cluster_id")
resumen.head(20)

## 5. Grupos frente a categorías

Cada fila es un grupo (los 18 más grandes) y cada columna una categoría de YouTube. El color es la fracción de los videos del grupo que cae en esa categoría: una fila con un solo cuadro oscuro es un grupo puro.

In [ ]:
top_grupos = resumen.index[:18]
top_cats = pd.Series(y[valido]).value_counts().index[:10]
M = (tabla.div(tabla.sum(axis=1), axis=0)).reindex(index=top_grupos, columns=top_cats).fillna(0)

fig, ax = plt.subplots(figsize=(12, 8))
im = ax.imshow(M.to_numpy(), cmap="YlGnBu", aspect="auto", vmin=0, vmax=1)
ax.set_xticks(range(len(top_cats)), top_cats, rotation=32, ha="right", fontsize=9.5)
ax.set_yticks(range(len(top_grupos)), [f"C{k:02d} · {resumen.nombre[k][:34]}" for k in top_grupos], fontsize=9)
for i in range(M.shape[0]):
    for j in range(M.shape[1]):
        v = M.iat[i, j]
        if v >= 0.05:
            ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=8, color="white" if v > 0.52 else INK)
ax.set_title("Qué fracción de cada grupo cae en cada categoría", loc="left", fontweight="bold", color=INK)
fig.colorbar(im, ax=ax, fraction=0.025, pad=0.02)
plt.tight_layout()
plt.show()

## 6. Cómo se reparten las categorías en grupos

El mismo cruce como diagrama de flujo: a la izquierda las 8 categorías más grandes, a la derecha los 12 grupos más grandes. Se ve cómo una categoría amplia se abre en varios temas.

In [ ]:
top_cats8 = top_cats[:8]
top_grupos12 = resumen.index[:12]
flujo = pd.crosstab(y, asignado).reindex(index=top_cats8, columns=top_grupos12).fillna(0)

etiquetas = list(top_cats8) + [f"C{k:02d} · {resumen.nombre[k]}" for k in top_grupos12]
origen, destino, valor = [], [], []
for i, cat in enumerate(top_cats8):
    for j, k in enumerate(top_grupos12):
        if flujo.loc[cat, k] >= 60:           # se omiten los flujos chicos para que se lea
            origen.append(i)
            destino.append(len(top_cats8) + j)
            valor.append(int(flujo.loc[cat, k]))

go.Figure(go.Sankey(node=dict(label=etiquetas, pad=12, thickness=14),
                    link=dict(source=origen, target=destino, value=valor))
          ).update_layout(title="Categorías de YouTube → grupos HDBSCAN", height=650, font_size=11)

## 7. El mapa coloreado por grupo

El mapa de la web, ahora coloreado por los grupos encontrados (los 20 más grandes; el resto queda gris como "otros").

In [ ]:
nombre_grupo = {k: f"C{k:02d} · {resumen.nombre[k]}" for k in resumen.index[:20]}
mapa = df[["title", "x", "y", "z"]].rename(columns={"x": "grupo_x", "y": "grupo_y", "z": "grupo_z"})
mapa["grupo"] = [nombre_grupo.get(k, "otros") for k in asignado]
plot3d(mapa, "grupo", "Mapa de la web · grupos HDBSCAN", color="grupo", leyenda="Grupo")

## Para llevar

- HDBSCAN encuentra temas más finos que las categorías de YouTube, y el bautizo zero-shot les pone nombres legibles sin intervención manual.
- La tabla queda en `data/hdbscan_centroids.csv` (nombre, tamaño, categoría dominante y pureza de cada grupo).